In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("glass.csv")

print("\nFirst 5 rows:\n")
print(df.head())

print("\nShape:", df.shape)
print("Columns:", df.columns.tolist())


First 5 rows:

        RI     Na    Mg    Al     Si     K    Ca   Ba   Fe  Type
0  1.52101  13.64  4.49  1.10  71.78  0.06  8.75  0.0  0.0     1
1  1.51761  13.89  3.60  1.36  72.73  0.48  7.83  0.0  0.0     1
2  1.51618  13.53  3.55  1.54  72.99  0.39  7.78  0.0  0.0     1
3  1.51766  13.21  3.69  1.29  72.61  0.57  8.22  0.0  0.0     1
4  1.51742  13.27  3.62  1.24  73.08  0.55  8.07  0.0  0.0     1

Shape: (214, 10)
Columns: ['RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe', 'Type']


In [3]:
# Output column is "Type"
# All other columns are numeric measurements
# If an ID column exists, we remove it

if "Id" in df.columns:
    df = df.drop(columns=["Id"])

In [4]:
# Type 1 = positive class
# All others = negative

df["y"] = (df["Type"] == 1).astype(int)

df = df.drop(columns=["Type"])

print("\nBinary label created.")
print(df["y"].value_counts())



Binary label created.
y
0    144
1     70
Name: count, dtype: int64


In [5]:
X = df.drop(columns=["y"]).values
y = df["y"].values

print("\nShape of X:", X.shape)
print("Shape of y:", y.shape)



Shape of X: (214, 9)
Shape of y: (214,)


In [6]:
N = len(X)
indices = np.random.permutation(N)

split = int(0.8 * N)

train_idx = indices[:split]
test_idx = indices[split:]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

In [7]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

# Scaling prevents sigmoid saturation.

In [8]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [9]:
def predict_proba(X, w, b):
    z = X @ w + b
    p = sigmoid(z)
    return p

In [10]:
def loss(y, p):
    eps = 1e-8  # prevent log(0)
    p = np.clip(p, eps, 1 - eps)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

# Confident wrong predictions get heavy penalty.

In [11]:
def update_weights(X, y, w, b, lr):
    
    p = predict_proba(X, w, b)
    
    error = p - y
    
    w = w - lr * (X.T @ error) / len(y)
    b = b - lr * np.mean(error)
    
    return w, b

In [ ]:
#Training
w = np.zeros(X_train.shape[1])
b = 0.0

lr = 0.1
epochs = 200

for epoch in range(epochs):
    
    w, b = update_weights(X_train, y_train, w, b, lr)
    
    if epoch % 20 == 0:
        p_train = predict_proba(X_train, w, b)
        current_loss = loss(y_train, p_train)
        print(f"Epoch {epoch}, Loss: {current_loss}")

Epoch 0, Loss: 0.6810827584620708
Epoch 20, Loss: 0.5667807889705313
Epoch 40, Loss: 0.5274014047311294
Epoch 60, Loss: 0.506039935871421
Epoch 80, Loss: 0.4922799393194602
Epoch 100, Loss: 0.4825534828459242
Epoch 120, Loss: 0.4752556145286653
Epoch 140, Loss: 0.46954841175125284
Epoch 160, Loss: 0.4649485187090898
Epoch 180, Loss: 0.4611556654002523


In [15]:
def predict_label(p, threshold=0.5):
    return (p >= threshold).astype(int)

In [16]:
p_test = predict_proba(X_test, w, b)

pred_05 = predict_label(p_test, 0.5)
pred_07 = predict_label(p_test, 0.7)

accuracy_05 = np.mean(pred_05 == y_test)
accuracy_07 = np.mean(pred_07 == y_test)

print("Test Accuracy (threshold=0.5):", accuracy_05)
print("Test Accuracy (threshold=0.7):", accuracy_07)

# Higher threshold (0.7) is safer in glass quality control
# because it reduces false positives.

Test Accuracy (threshold=0.5): 0.6976744186046512
Test Accuracy (threshold=0.7): 0.7674418604651163
